<a href="https://colab.research.google.com/github/mythri2506a/amazon_ml_challenge/blob/main/Basic_blocking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon ML Challenge — blocking only



It selects a reproducible Source 1 sample, adds all known positive Source 2/3 records for those entities to a bounded candidate pool, builds a country and token index, generates top-K candidates, and measures **conditional blocking recall on that sampled pool**. These numbers are diagnostic, not a full-data or test-set score. It writes reports and candidate pairs under `MyDrive/Amazon_ML_Challenge`.

In [1]:
# 1 — Colab dependencies and Drive mount
%pip -q install pandas pyarrow
from google.colab import drive
drive.mount('/content/drive')

import math
import re
import time
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display


Mounted at /content/drive


In [2]:
# 2 — Input paths, output paths, and memory budget
MY_DRIVE = Path('/content/drive/MyDrive')
TRAIN_DIR = MY_DRIVE / 'Amazon_Ml_Challange_2026' / 'Data' / 'Processed' / 'dataset' / 'train'
OUT_ROOT = MY_DRIVE / 'Amazon_ML_Challenge'
REPORT_DIR = OUT_ROOT / 'reports'
CANDIDATE_DIR = OUT_ROOT / 'candidates'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_S1 = TRAIN_DIR / 'train_source1.tsv'
TRAIN_S2 = TRAIN_DIR / 'train_source2.tsv'
TRAIN_S3 = TRAIN_DIR / 'train_source3.tsv'
TRAIN_GT = TRAIN_DIR / 'train_ground_truth.tsv'
for path in (TRAIN_S1, TRAIN_S2, TRAIN_S3, TRAIN_GT):
    if not path.is_file():
        raise FileNotFoundError(
            f'Missing {path}. Run the extraction notebook first and ensure the shared folder '
            'has a shortcut in My Drive, or edit TRAIN_DIR.'
        )

SAMPLE_S1 = 5_000          # Reduce to 1_000 if RAM is tight
DISTRACTORS_PER_SOURCE = 50_000  # Plus all known positives for selected S1
CHUNK_ROWS = 50_000
TOP_K = 40
MAX_DF = 400
assert SAMPLE_S1 > 0 and DISTRACTORS_PER_SOURCE > 0 and CHUNK_ROWS > 0 and TOP_K > 0
print('Reading:', TRAIN_DIR)
print('Saving reports:', REPORT_DIR)
print('Source 1 sample:', SAMPLE_S1, '| distractors/source:', DISTRACTORS_PER_SOURCE)


Reading: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/train
Saving reports: /content/drive/MyDrive/Amazon_ML_Challenge/reports
Source 1 sample: 5000 | distractors/source: 50000


## Sample Source 1 and read its ground truth

The sampler keeps the records with the lowest stable ID hashes. It processes one TSV chunk at a time and produces the same sample on reruns with the same settings.

In [3]:
# 3 — Chunked, deterministic Source 1 sample
SOURCE_COLS = ['entity_id', 'business_name', 'business_address', 'country']

def read_chunks(path, usecols, chunksize=CHUNK_ROWS):
    return pd.read_csv(path, sep='\t', usecols=usecols, dtype=str,
                       keep_default_na=False, chunksize=chunksize)

def keep_lowest_hash(current, incoming, limit):
    if incoming.empty:
        return current
    incoming = incoming.copy()
    incoming['_rank'] = pd.util.hash_pandas_object(incoming['entity_id'], index=False).to_numpy()
    incoming = incoming.nsmallest(limit, '_rank')
    combined = pd.concat([current, incoming], ignore_index=True) if current is not None else incoming
    return combined.nsmallest(limit, '_rank').reset_index(drop=True)

start = time.time()
dev_s1 = None
seen_s1 = 0
for chunk in read_chunks(TRAIN_S1, SOURCE_COLS):
    seen_s1 += len(chunk)
    dev_s1 = keep_lowest_hash(dev_s1, chunk, SAMPLE_S1)
if dev_s1 is None or dev_s1.empty:
    raise ValueError('Train Source 1 is empty.')
dev_s1 = dev_s1.drop(columns='_rank').drop_duplicates('entity_id').reset_index(drop=True)
selected_s1_ids = set(dev_s1['entity_id'])
print(f'Scanned {seen_s1:,} S1 rows; sampled {len(dev_s1):,} in {time.time()-start:.1f}s')
display(dev_s1.head(3))


Scanned 2,206,821 S1 rows; sampled 5,000 in 12.0s


,entity_id,business_name,business_address,country
0,S1-385865318,Life Prime Investments Private Limited,"House No 07, Pocket D3, Second Floor Rohini, S...",India
1,S1-419256346,Walker Learning Center LLC,"719 Carbon Avenue, Hartshorne, OK",US
2,S1-756569582,Carmon Martinez Kongsberg LLC,"505 Hobson Place, Pittsburg, KS",US


In [4]:
# 4 — Read only ground-truth rows for selected Source 1 IDs
def parse_ids(value):
    return [item.strip() for item in value.split(',') if item.strip()]

truth = {}
for chunk in read_chunks(TRAIN_GT, ['source1_entity_id', 'matched_entity_ids']):
    rows = chunk[chunk['source1_entity_id'].isin(selected_s1_ids)]
    for source1_id, targets in rows.itertuples(index=False, name=None):
        if source1_id in truth:
            raise ValueError(f'Duplicate ground-truth row for {source1_id}')
        truth[source1_id] = set(parse_ids(targets))
missing_truth = selected_s1_ids - set(truth)
if missing_truth:
    raise ValueError(f'{len(missing_truth)} sampled S1 entities have no ground-truth row.')
positive_ids = set().union(*truth.values()) if truth else set()
print('Sampled S1:', len(truth))
print('Sampled singletons:', sum(not targets for targets in truth.values()))
print('Distinct known matching S2/S3 IDs:', len(positive_ids))
print('True pairs:', sum(map(len, truth.values())))


Sampled S1: 5000
Sampled singletons: 262
Distinct known matching S2/S3 IDs: 17513
True pairs: 17513


## Candidate pool and index

The pool contains every true target ID found for the sampled Source 1 records, plus up to the configured number of distractors from each target source. This deliberate inclusion makes the subsequent recall estimate conditional on the pool. It prevents random sampling from hiding genuine blocking misses.

In [5]:
# 5 — Chunked target-source sampling, retaining known positives

def sample_target_source(path, required_ids, distractor_limit):
    best = None
    positive_chunks = []
    total_rows = 0
    for chunk in read_chunks(path, SOURCE_COLS):
        total_rows += len(chunk)
        needed = chunk['entity_id'].isin(required_ids)
        if needed.any():
            positive_chunks.append(chunk.loc[needed].copy())
        best = keep_lowest_hash(best, chunk.loc[~needed], distractor_limit)
    positives = (pd.concat(positive_chunks, ignore_index=True)
                 if positive_chunks else pd.DataFrame(columns=SOURCE_COLS))
    distractors = (best.drop(columns='_rank') if best is not None
                   else pd.DataFrame(columns=SOURCE_COLS))
    pool = pd.concat([positives, distractors], ignore_index=True)
    pool = pool.drop_duplicates('entity_id').reset_index(drop=True)
    print(f'{path.name}: scanned {total_rows:,}; positives {len(positives):,}; pool {len(pool):,}')
    return pool

start = time.time()
dev_s2 = sample_target_source(TRAIN_S2, positive_ids, DISTRACTORS_PER_SOURCE)
dev_s3 = sample_target_source(TRAIN_S3, positive_ids, DISTRACTORS_PER_SOURCE)
candidate_pool = pd.concat([dev_s2, dev_s3], ignore_index=True)
if candidate_pool['entity_id'].duplicated().any():
    raise ValueError('Duplicate IDs across target sources; inspect the files.')
known_in_pool = positive_ids & set(candidate_pool['entity_id'])
if missing := positive_ids - known_in_pool:
    print(f'WARNING: {len(missing)} ground-truth target IDs were not found in train S2/S3.')
print(f'Candidate pool: {len(candidate_pool):,}; elapsed: {time.time()-start:.1f}s')
del dev_s2, dev_s3


train_source2.tsv: scanned 5,034,616; positives 8,469; pool 58,469
train_source3.tsv: scanned 5,285,603; positives 9,044; pool 59,044
Candidate pool: 117,513; elapsed: 49.8s


In [6]:
# 6 — Normalize names/addresses and define blocking keys
LEGAL = {
    'pvt': 'private', 'pvte': 'private', 'ltd': 'limited', 'lim': 'limited',
    'corp': 'corporation', 'inc': 'incorporated', 'co': 'company', 'cos': 'company',
}
ADDR_ABBR = {
    'rd': 'road', 'st': 'street', 'str': 'street', 'ave': 'avenue',
    'av': 'avenue', 'blvd': 'boulevard', 'ln': 'lane', 'dr': 'drive',
    'ct': 'court', 'pkwy': 'parkway', 'hwy': 'highway', 'nr': 'near',
    'opp': 'opposite', 'apt': 'apartment', 'ste': 'suite', 'fl': 'floor',
    'flr': 'floor', 'bldg': 'building', 'blk': 'block', 'no': 'number',
    'nos': 'number', 'hno': 'number', 'sec': 'sector',
}
STOP = {
    'private','limited','incorporated','corporation','company','llp','llc','plc',
    'the','and','of','number','road','street','avenue','lane','drive',
    'near','floor','block','sector'
}
PUNCT = re.compile(r'[^\w\s]', re.UNICODE)
SPACE = re.compile(r'\s+')
ORDINAL = re.compile(r'^(\d+)(st|nd|rd|th)$')

def base_norm(value):
    value = unicodedata.normalize('NFKD', value or '')
    value = ''.join(ch for ch in value if not unicodedata.combining(ch))
    value = PUNCT.sub(' ', value.casefold().replace('&', ' and '))
    return SPACE.sub(' ', value).strip()

def keys_for(name, address):
    keys = set()
    for token in base_norm(name).split():
        token = LEGAL.get(token, token)
        if len(token) >= 3 and token not in STOP:
            keys.add('n|' + token)
        if len(token) >= 5 and token not in STOP:
            keys.add('p|' + token[:4])
    for token in base_norm(address).split():
        ordinal = ORDINAL.match(token)
        if ordinal:
            token = ordinal.group(1)
        token = ADDR_ABBR.get(token, token)
        if token.isdigit() and len(token) >= 2:
            keys.add('d|' + token)
        elif len(token) >= 4 and token not in STOP and not token.isdigit():
            keys.add('a|' + token)
        else:
            number = re.match(r'^(\d{2,})', token)
            if number:
                keys.add('d|' + number.group(1))
    return keys

print('Example name keys:', sorted(keys_for('ABC Technologies Pvt Ltd', '12 M G Road'))[:10])


Example name keys: ['d|12', 'n|abc', 'n|technologies', 'p|tech']


In [7]:
# 7 — Build country-specific inverted index; drop very common keys
start = time.time()
index = defaultdict(lambda: defaultdict(list))
pool_ids = candidate_pool['entity_id'].tolist()
country_pool_sizes = candidate_pool['country'].value_counts().to_dict()
for row_idx, (country, name, address) in enumerate(
    candidate_pool[['country','business_name','business_address']].itertuples(index=False, name=None)
):
    for key in keys_for(name, address):
        index[country][key].append(row_idx)
    if row_idx and row_idx % 50_000 == 0:
        print(f'Indexed {row_idx:,}/{len(candidate_pool):,}')

index_stats = []
for country, postings in index.items():
    dropped = [key for key, rows in postings.items() if len(rows) > MAX_DF]
    for key in dropped:
        del postings[key]
    index_stats.append({'country': country, 'keys_kept': len(postings),
                        'keys_dropped_too_common': len(dropped)})
display(pd.DataFrame(index_stats))
print(f'Index ready in {time.time()-start:.1f}s')


Indexed 50,000/117,513
Indexed 100,000/117,513


,country,keys_kept,keys_dropped_too_common
0,India,68632,158
1,US,87953,137


Index ready in 3.8s


## Generate candidates and evaluate

Each sampled Source 1 record gets at most `TOP_K` candidate IDs. Zero-candidate cases, missed true pairs, country exclusions, and candidate reduction are reported. A strong conditional result still needs confirmation on a larger or complete pool.

In [8]:
# 8 — Score rare shared keys and keep top-K candidates
start = time.time()
candidates = {}
for number, row in enumerate(dev_s1.itertuples(index=False), 1):
    postings = index.get(row.country, {})
    scores = defaultdict(float)
    n_country = max(country_pool_sizes.get(row.country, 1), 1)
    for key in keys_for(row.business_name, row.business_address):
        target_rows = postings.get(key, [])
        if target_rows:
            weight = math.log1p(n_country / len(target_rows))
            for row_idx in target_rows:
                scores[row_idx] += weight
    ranked = sorted(scores.items(), key=lambda item: (-item[1], pool_ids[item[0]]))[:TOP_K]
    candidates[row.entity_id] = [pool_ids[row_idx] for row_idx, _ in ranked]
    if number % 1_000 == 0:
        print(f'Generated candidates for {number:,}/{len(dev_s1):,} S1 records')
print(f'Done in {time.time()-start:.1f}s')


Generated candidates for 1,000/5,000 S1 records
Generated candidates for 2,000/5,000 S1 records
Generated candidates for 3,000/5,000 S1 records
Generated candidates for 4,000/5,000 S1 records
Generated candidates for 5,000/5,000 S1 records
Done in 4.0s


In [9]:
# 9 — Conditional recall and candidate efficiency
reachable = set(pool_ids)
country_for_target = dict(zip(candidate_pool['entity_id'], candidate_pool['country']))
s1_country = dict(zip(dev_s1['entity_id'], dev_s1['country']))
true_pairs = hits = missed = missing_targets = cross_country = zero_candidates = total_candidates = 0
entity_recalls = []
missed_rows = []
for source1_id in dev_s1['entity_id']:
    gold = truth[source1_id]
    prediction = set(candidates[source1_id])
    total_candidates += len(prediction)
    zero_candidates += not prediction
    missing_targets += len(gold - reachable)
    known = gold & reachable
    cross_country += sum(country_for_target[target] != s1_country[source1_id] for target in known)
    true_pairs += len(known)
    found = len(known & prediction)
    hits += found
    missed += len(known) - found
    if known:
        entity_recalls.append(found / len(known))
    for target in sorted(known - prediction):
        if len(missed_rows) < 100:
            missed_rows.append({'source1_entity_id': source1_id,
                                'missed_true_match_id': target,
                                'cross_country': country_for_target[target] != s1_country[source1_id]})

avg_candidates = total_candidates / len(dev_s1)
summary = pd.DataFrame([{
    'sample_s1': len(dev_s1), 'sample_pool_s2_s3': len(candidate_pool),
    'known_true_pairs_in_pool': true_pairs, 'unknown_target_ids': missing_targets,
    'cross_country_true_pairs_in_pool': cross_country,
    'top_k': TOP_K, 'max_df': MAX_DF,
    'conditional_pair_recall': hits / true_pairs if true_pairs else float('nan'),
    'conditional_macro_entity_recall': sum(entity_recalls) / len(entity_recalls) if entity_recalls else float('nan'),
    'avg_candidates_per_s1': avg_candidates,
    'sample_pool_reduction_ratio': 1 - avg_candidates / len(candidate_pool),
    'zero_candidate_rate': zero_candidates / len(dev_s1),
}])
display(summary.T.rename(columns={0: 'value'}))
print('Missed pairs:', missed, '| zero-candidate S1:', zero_candidates)


,value
sample_s1,5000.000000
sample_pool_s2_s3,117513.000000
known_true_pairs_in_pool,17513.000000
unknown_target_ids,0.000000
cross_country_true_pairs_in_pool,0.000000
top_k,40.000000
max_df,400.000000
conditional_pair_recall,0.969508
conditional_macro_entity_recall,0.969536
avg_candidates_per_s1,39.833800


Missed pairs: 534 | zero-candidate S1: 0


In [10]:
# 10 — Save the diagnostic outputs to Drive (not final submission files)
label = f's1{len(dev_s1)}_pool{len(candidate_pool)}_k{TOP_K}_df{MAX_DF}'
summary_path = REPORT_DIR / f'blocking_summary_{label}.csv'
missed_path = REPORT_DIR / f'blocking_missed_{label}.csv'
candidate_path = CANDIDATE_DIR / f'dev_candidate_pairs_{label}.parquet'
summary.to_csv(summary_path, index=False)
pd.DataFrame(missed_rows, columns=['source1_entity_id','missed_true_match_id','cross_country']).to_csv(missed_path, index=False)
# Store flat sampled pairs; the final challenge submission needs a separate test-set TSV
# with one row per test S1, so do not upload this development artifact to the portal.
pair_rows = [(source1_id, target_id)
             for source1_id, ids in candidates.items() for target_id in ids]
pd.DataFrame(pair_rows, columns=['source1_entity_id','candidate_entity_id']).to_parquet(candidate_path, index=False)
print('Saved summary:', summary_path)
print('Saved misses:', missed_path)
print('Saved sampled candidates:', candidate_path)


Saved summary: /content/drive/MyDrive/Amazon_ML_Challenge/reports/blocking_summary_s15000_pool117513_k40_df400.csv
Saved misses: /content/drive/MyDrive/Amazon_ML_Challenge/reports/blocking_missed_s15000_pool117513_k40_df400.csv
Saved sampled candidates: /content/drive/MyDrive/Amazon_ML_Challenge/candidates/dev_candidate_pairs_s15000_pool117513_k40_df400.parquet


In [11]:
# Diagnose the current blocker without rescanning the TSV files.
from collections import Counter, defaultdict
import math

pool_row = {entity_id: i for i, entity_id in enumerate(pool_ids)}
hit_counts = Counter()
miss_reasons = Counter()
total_true = 0

for row in dev_s1.itertuples(index=False):
    postings = index.get(row.country, {})
    scores = defaultdict(float)
    n_country = max(country_pool_sizes.get(row.country, 1), 1)

    for key in keys_for(row.business_name, row.business_address):
        target_rows = postings.get(key, [])
        if target_rows:
            weight = math.log1p(n_country / len(target_rows))
            for row_idx in target_rows:
                scores[row_idx] += weight

    ranked = sorted(
        scores,
        key=lambda i: (-scores[i], pool_ids[i])
    )
    rank = {pool_ids[i]: position for position, i in enumerate(ranked, 1)}

    for target_id in truth[row.entity_id]:
        if target_id not in pool_row:
            continue
        total_true += 1
        position = rank.get(target_id)

        if position is None:
            miss_reasons["No retained shared key"] += 1
        elif position > 160:
            miss_reasons["Ranked below 160"] += 1
        elif position > 80:
            miss_reasons["Ranked 81–160"] += 1
        elif position > 40:
            miss_reasons["Ranked 41–80"] += 1

        for k in (40, 80, 160):
            if position is not None and position <= k:
                hit_counts[k] += 1

print("True pairs in sampled pool:", total_true)
for k in (40, 80, 160):
    print(f"Recall at K={k}: {hit_counts[k] / total_true:.2%}")
print("Reasons pairs miss K=40:")
for reason, count in miss_reasons.items():
    print(f"  {reason}: {count}")

True pairs in sampled pool: 17513
Recall at K=40: 96.95%
Recall at K=80: 97.37%
Recall at K=160: 98.00%
Reasons pairs miss K=40:
  Ranked below 160: 230
  No retained shared key: 120
  Ranked 41–80: 74
  Ranked 81–160: 110
